In [3]:
from google_play_scraper import reviews, Sort
import pandas as pd

def scrape_app_reviews(app_id, app_name, count=500):
    result, _ = reviews(
        app_id,
        lang='en',
        country='us',
        sort=Sort.NEWEST,
        count=count
    )
    data = []
    for r in result:
        data.append({
            'review': r['content'],
            'rating': r['score'],
            'date': r['at'],
            'bank': app_name,
            'source': 'Google Play'
        })
    return pd.DataFrame(data)

apps = [
    ('com.combanketh.mobilebanking', 'Commercial Bank of Ethiopia'),
    ('com.boa.boaMobileBanking', 'Bank of Abyssinia'),
    ('com.cr2.amolelight', 'Dashen Bank')
]

df_list = []
for app_id, app_name in apps:
    print(f"Scraping {app_name}...")
    df = scrape_app_reviews(app_id, app_name, count=500)
    df_list.append(df)
    print(f"  -> got {len(df)} reviews")

all_reviews = pd.concat(df_list, ignore_index=True)
print(f"\nTotal reviews scraped: {len(all_reviews)}")
all_reviews.head()

Scraping Commercial Bank of Ethiopia...
  -> got 500 reviews
Scraping Bank of Abyssinia...
  -> got 500 reviews
Scraping Dashen Bank...
  -> got 500 reviews

Total reviews scraped: 1500


,review,rating,date,bank,source
0,formative,5,2026-05-14 16:49:46,Commercial Bank of Ethiopia,Google Play
1,best app for financial activities 🙌,5,2026-05-14 16:46:29,Commercial Bank of Ethiopia,Google Play
2,yoroo namaste 🙏 ♥️ ❤️ 💖 💖,5,2026-05-14 12:13:13,Commercial Bank of Ethiopia,Google Play
3,incredible,5,2026-05-14 10:53:56,Commercial Bank of Ethiopia,Google Play
4,best app for financial sector,5,2026-05-14 05:44:01,Commercial Bank of Ethiopia,Google Play


In [4]:
import os
os.makedirs('data/raw', exist_ok=True)
all_reviews.to_csv('data/raw/raw_reviews.csv', index=False)
print("Raw data saved to data/raw/raw_reviews.csv")

Raw data saved to data/raw/raw_reviews.csv


In [5]:
df = all_reviews.copy()

# Remove duplicates
initial = len(df)
df = df.drop_duplicates(subset='review')
print(f"Removed {initial - len(df)} duplicates")

# Drop rows missing review text or rating
initial = len(df)
df = df.dropna(subset=['review', 'rating'])
print(f"Dropped {initial - len(df)} rows with missing data")

# Normalise date to YYYY-MM-DD
df['date'] = pd.to_datetime(df['date']).dt.date

# Ensure rating is integer
df['rating'] = df['rating'].astype(int)

# Select required columns
cleaned_df = df[['review', 'rating', 'date', 'bank', 'source']]

# Save cleaned CSV
os.makedirs('data/raw', exist_ok=True)
cleaned_df.to_csv('data/raw/cleaned_reviews.csv', index=False)
print(f"Cleaned dataset saved. Total reviews: {len(cleaned_df)}")
cleaned_df.head()

Removed 366 duplicates
Dropped 0 rows with missing data
Cleaned dataset saved. Total reviews: 1134


,review,rating,date,bank,source
0,formative,5,2026-05-14,Commercial Bank of Ethiopia,Google Play
1,best app for financial activities 🙌,5,2026-05-14,Commercial Bank of Ethiopia,Google Play
2,yoroo namaste 🙏 ♥️ ❤️ 💖 💖,5,2026-05-14,Commercial Bank of Ethiopia,Google Play
3,incredible,5,2026-05-14,Commercial Bank of Ethiopia,Google Play
4,best app for financial sector,5,2026-05-14,Commercial Bank of Ethiopia,Google Play
